# Upstream GR00T N1.6 RoboCasa primitive-policy suitability gate

**Scope:** Run NVIDIA Isaac-GR00T upstream `Gr00tSimPolicyWrapper` + RoboCasa `OpenDrawer_PandaOmron_Env` on Colab. This is **not** a Longor evaluation; **no result here supports any Longor claim**.

Docs: [RoboCasa example (n1d6)](https://github.com/NVIDIA/Isaac-GR00T/blob/n1d6/examples/robocasa/README.md), [Mintlify RoboCasa](https://nvidia-isaac-gr00t.mintlify.app/examples/robocasa).

## Instructions

1. Colab runtime: **Linux + NVIDIA GPU** (T4/A100/L4 class; 16GB+ VRAM recommended per upstream docs).
2. Optional: set Hugging Face token in Colab **Secrets** as `HF_TOKEN` if `nvidia/GR00T-N1.6-3B` download is gated.
3. Run all cells top to bottom.
4. Artifacts are written under **`/content/drive/MyDrive/Colab/longor_robocasa_primitive_gate/<UTC>/`** (timestamped; prior runs kept). M1 upstream `verdict.txt` is written under the artifact directory.

## Step 1 - Mount Drive and initialize artifact paths

Create the run artifact directory and set task constants for this gate attempt.

In [ ]:
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ARTIFACT_BASE = Path('/content/drive/MyDrive/Colab/longor_robocasa_primitive_gate')
ARTIFACT_DIR = ARTIFACT_BASE / RUN_UTC
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

GROOT_ROOT = Path('/content/Isaac-GR00T')
GROOT_BRANCH = 'n1d6'
MODEL_PATH = 'nvidia/GR00T-N1.6-3B'
EMBODIMENT_TAG = 'ROBOCASA_PANDA_OMRON'
ENV_NAME = 'robocasa_panda_omron/OpenDrawer_PandaOmron_Env'

for name in ('setup.log', 'policy_server.log', 'rollout.log', 'metadata.json', 'verdict.txt'):
    (ARTIFACT_DIR / name).touch(exist_ok=True)

print('ARTIFACT_DIR=', ARTIFACT_DIR)
print('RUN_UTC=', RUN_UTC)
print('ARTIFACT_BASE=', ARTIFACT_BASE)

## Step 2 - Runtime environment and optional Hugging Face auth

Export runtime env vars and login only if `HF_TOKEN` is available.

In [ ]:
import os
from pathlib import Path

os.environ['ARTIFACT_DIR'] = str(ARTIFACT_DIR)
os.environ['GROOT_ROOT'] = str(GROOT_ROOT)
os.environ['GROOT_BRANCH'] = GROOT_BRANCH
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

if os.environ.get('HF_TOKEN'):
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential

## Step 3 - Build upstream environment (dependency-safe path)

Clone Isaac-GR00T, install system deps, run upstream-style `uv sync`, and patch `setup_RoboCasa.sh` so `uv pip` targets the intended robocasa Python 3.10 venv.

In [ ]:
%%bash
set -uo pipefail
exec > >(tee "${ARTIFACT_DIR}/setup.log") 2>&1

echo "=== setup start $(date -u +%Y-%m-%dT%H:%M:%SZ) ==="
echo "GROOT_BRANCH=${GROOT_BRANCH}"

set -e
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  git git-lfs ffmpeg libaio-dev libegl1-mesa-dev libglu1-mesa
git lfs install

if [ ! -d "${GROOT_ROOT}/.git" ]; then
  git clone --branch "${GROOT_BRANCH}" --recurse-submodules \
    https://github.com/NVIDIA/Isaac-GR00T.git "${GROOT_ROOT}"
else
  git -C "${GROOT_ROOT}" fetch origin "${GROOT_BRANCH}"
  git -C "${GROOT_ROOT}" checkout "${GROOT_BRANCH}"
  git -C "${GROOT_ROOT}" submodule update --init --recursive
fi

cd "${GROOT_ROOT}"
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"

cat > "${ARTIFACT_DIR}/resolver_hotfix.txt" <<'EOF'
Root cause A: NVIDIA n1d6 install_deps.sh runs `uv pip install -e .` after `uv sync`.
That second step re-resolves dependencies and can currently conflict on:
- gr00t pins numpy==1.26.4
- gr00t requires onnx>=1.20.0
- newest onnx wheels require ml-dtypes>=0.5.0 -> numpy>=2.1.0
Fix A in this notebook: keep upstream-style `uv sync` but skip the extra
editable reinstall step.

Root cause B (previous attempt): adding `#sha256=` URL fragments inside
`pyproject.toml` direct references is rejected by uv for this format.
Fix B in this notebook: do not rewrite direct URLs; rely on normal `uv sync`.

Root cause C (current): setup_RoboCasa.sh executes `uv pip install` steps against
system Python (/usr, 3.13) instead of robocasa_uv/.venv (3.10), causing
`torchvision==0.20.1` wheel ABI mismatch.
Fix C in this notebook: patch setup_RoboCasa.sh in-place so every `uv pip install`
explicitly targets `--python "$UV_ENV/.venv/bin/python"`.
EOF

python3 - <<'PY'
from pathlib import Path

script = Path('gr00t/eval/sim/robocasa/setup_RoboCasa.sh')
text = script.read_text(encoding='utf-8')
needle = 'uv pip install '
patched = 'uv pip install --python "$UV_ENV/.venv/bin/python" '
if patched not in text:
    text = text.replace(needle, patched)
    script.write_text(text, encoding='utf-8')
    print('Patched setup_RoboCasa.sh to force uv pip --python $UV_ENV/.venv/bin/python')
else:
    print('setup_RoboCasa.sh already patched')
PY

# Keep install flow close to upstream, but avoid the extra re-resolve path.
uv sync
source .venv/bin/activate
bash gr00t/eval/sim/robocasa/setup_RoboCasa.sh

echo "=== setup ok $(date -u +%Y-%m-%dT%H:%M:%SZ) ==="

## Step 4 - Capture setup provenance metadata

Record SHAs, package versions, runtime details, and the exact rollout command reference.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

def git_head(repo: Path) -> str | None:
    if not (repo / '.git').exists():
        return None
    out = subprocess.run(
        ['git', '-C', str(repo), 'rev-parse', 'HEAD'],
        check=False,
        capture_output=True,
        text=True,
    )
    sha = out.stdout.strip()
    return sha if out.returncode == 0 and sha else None


def robocasa_venv_python(groot: Path) -> Path:
    return groot / 'gr00t/eval/sim/robocasa/robocasa_uv/.venv/bin/python'


def package_version(python_bin: Path, package_name: str) -> str:
    """Query package version from a target interpreter without pip."""
    script = (
        "import importlib.metadata as md, sys\n"
        "name = sys.argv[1]\n"
        "try:\n"
        "    print(md.version(name))\n"
        "except md.PackageNotFoundError:\n"
        "    print('NOT_INSTALLED')\n"
    )
    proc = subprocess.run(
        [str(python_bin), '-c', script, package_name],
        check=False,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        return f'QUERY_FAILED(exit={proc.returncode})'
    return proc.stdout.strip() or 'UNKNOWN'


def pip_versions(groot: Path) -> dict[str, str]:
    py = robocasa_venv_python(groot)
    names = [
        'torch',
        'transformers',
        'gymnasium',
        'robosuite',
        'robocasa',
        'numpy',
        'mujoco',
    ]
    return {name: package_version(py, name) for name in names}


def query_gpu() -> str:
    """GPU name/driver/memory, or a note when nvidia-smi is absent."""
    try:
        proc = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        return 'nvidia-smi not found'
    if proc.returncode == 0:
        return proc.stdout.strip()
    return proc.stderr.strip() or f'nvidia-smi exit {proc.returncode}'


groot = Path(os.environ['GROOT_ROOT'])
robocasa_repo = groot / 'external_dependencies' / 'robocasa'
robosuite_git = None
robosuite_site = groot / 'gr00t/eval/sim/robocasa/robocasa_uv/.venv/lib'
for site in robosuite_site.glob('python*/site-packages'):
    for direct in site.glob('robosuite-*.dist-info/direct_url.json'):
        try:
            robosuite_git = json.loads(direct.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            robosuite_git = None

venv_versions = pip_versions(groot)

metadata = {
    'artifact_dir': str(ARTIFACT_DIR),
    'run_utc': RUN_UTC,
    'groot_branch': GROOT_BRANCH,
    'groot_head': git_head(groot),
    'robocasa_head': git_head(robocasa_repo),
    'robosuite_pip_version': venv_versions.get('robosuite'),
    'robosuite_direct_url': robosuite_git,
    'python': sys.version,
    'robocasa_venv_python': str(robocasa_venv_python(groot)),
    'pip_packages_robocasa_venv': venv_versions,
    'gpu': query_gpu(),
    'model_path': MODEL_PATH,
    'embodiment_tag': EMBODIMENT_TAG,
    'env_name': ENV_NAME,
    'rollout_cli_reference': (
        'gr00t/eval/sim/robocasa/robocasa_uv/.venv/bin/python gr00t/eval/rollout_policy.py '
        '--n_episodes 10 --policy_client_host 127.0.0.1 --policy_client_port 5555 '
        '--max_episode_steps=720 --env_name robocasa_panda_omron/OpenDrawer_PandaOmron_Env '
        '--n_action_steps 8 --n_envs 5'
    ),
}

meta_text = json.dumps(metadata, indent=2) + '\n'
meta_path = ARTIFACT_DIR / 'metadata.json'
meta_path.write_text(meta_text, encoding='utf-8')
(ARTIFACT_DIR / 'metadata.log').write_text(meta_text, encoding='utf-8')
print(meta_text)

## Step 5 - Start policy server and wait for readiness

Launch `run_gr00t_server.py`, then gate on upstream readiness evidence (log marker + loopback port).

In [ ]:
import json
import os
import socket
import subprocess
import time
from pathlib import Path

POLICY_PORT = 5555
READINESS_LOG_MARKER = 'Server is ready and listening on'
READINESS_TIMEOUT_S = 900
POLL_INTERVAL_S = 5
MIN_PORT_READY_POLLS = 3


def port_accepts_loopback(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1.0)
        return sock.connect_ex(('127.0.0.1', port)) == 0


groot = Path(os.environ['GROOT_ROOT'])
server_log = ARTIFACT_DIR / 'policy_server.log'
server_log.write_text('', encoding='utf-8')

env = os.environ.copy()
env['MUJOCO_GL'] = 'egl'
env['PYOPENGL_PLATFORM'] = 'egl'
env['PYTHONUNBUFFERED'] = '1'

cmd = (
    f'cd "{groot}" && source .venv/bin/activate && '
    f'uv run python gr00t/eval/run_gr00t_server.py '
    f'--model-path {MODEL_PATH} '
    f'--embodiment-tag {EMBODIMENT_TAG} '
    f'--use-sim-policy-wrapper'
)

started = time.monotonic()
with server_log.open('ab') as log_fp:
    proc = subprocess.Popen(
        ['bash', '-lc', cmd],
        stdout=log_fp,
        stderr=subprocess.STDOUT,
        env=env,
    )

SERVER_PID = proc.pid
print('policy_server_pid=', SERVER_PID)

ready = False
reason = ''
port_ready_streak = 0
deadline = time.monotonic() + READINESS_TIMEOUT_S
while time.monotonic() < deadline:
    if proc.poll() is not None:
        reason = f'policy server exited before readiness (exit={proc.returncode})'
        break
    log_text = server_log.read_text(encoding='utf-8', errors='replace')
    log_ready = READINESS_LOG_MARKER in log_text
    port_ready = port_accepts_loopback(POLICY_PORT)
    port_ready_streak = port_ready_streak + 1 if port_ready else 0
    if log_ready and port_ready:
        ready = True
        reason = (
            f'upstream log marker {READINESS_LOG_MARKER!r} and '
            f'127.0.0.1:{POLICY_PORT} accept'
        )
        break
    if port_ready_streak >= MIN_PORT_READY_POLLS:
        ready = True
        reason = (
            f'127.0.0.1:{POLICY_PORT} accepted connections for '
            f'{port_ready_streak} consecutive polls; '
            'readiness log marker may be buffered'
        )
        break
    time.sleep(POLL_INTERVAL_S)

if not ready and not reason:
    log_text = server_log.read_text(encoding='utf-8', errors='replace')
    reason = (
        f'readiness timeout after {READINESS_TIMEOUT_S}s: '
        f'log_marker_seen={READINESS_LOG_MARKER in log_text}; '
        f'port_{POLICY_PORT}_open={port_accepts_loopback(POLICY_PORT)}; '
        f'port_ready_streak={port_ready_streak}'
    )

readiness = {
    'ready': ready,
    'reason': reason,
    'timeout_s': READINESS_TIMEOUT_S,
    'port': POLICY_PORT,
    'log_marker': READINESS_LOG_MARKER,
    'min_port_ready_polls': MIN_PORT_READY_POLLS,
    'final_port_ready_streak': port_ready_streak,
    'elapsed_s': round(time.monotonic() - started, 2),
    'server_pid': SERVER_PID,
}
readiness_path = ARTIFACT_DIR / 'server_readiness.json'
readiness_path.write_text(json.dumps(readiness, indent=2) + '\n', encoding='utf-8')
print(json.dumps(readiness, indent=2))
print(server_log.read_text(encoding='utf-8', errors='replace')[-4000:])

## Step 6 - Run upstream rollout client

Execute the official RoboCasa rollout command against the local policy server and capture the real exit code.

In [ ]:
%%bash
set -uo pipefail
exec > >(tee -a "${ARTIFACT_DIR}/rollout.log") 2>&1

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
cd "${GROOT_ROOT}"

ready="$(python3 - <<'PY'
import json
import os
print(json.load(open(os.environ['ARTIFACT_DIR'] + '/server_readiness.json'))['ready'])
PY
)"
if [ "${ready}" != "True" ]; then
  skip_reason="$(python3 - <<'PY'
import json
import os
print(json.load(open(os.environ['ARTIFACT_DIR'] + '/server_readiness.json'))['reason'])
PY
)"
  echo "=== rollout skipped: policy server not ready ==="
  echo "${skip_reason}"
  echo "-1" > "${ARTIFACT_DIR}/rollout_exit_code.txt"
  echo "=== rollout finished $(date -u +%Y-%m-%dT%H:%M:%SZ) skipped=1 ==="
  exit 0
fi

set +e
gr00t/eval/sim/robocasa/robocasa_uv/.venv/bin/python gr00t/eval/rollout_policy.py \
  --n_episodes 10 \
  --policy_client_host 127.0.0.1 \
  --policy_client_port 5555 \
  --max_episode_steps=720 \
  --env_name robocasa_panda_omron/OpenDrawer_PandaOmron_Env \
  --n_action_steps 8 \
  --n_envs 5
rollout_exit_code=$?
set -e
echo "=== rollout exit code: ${rollout_exit_code} ==="
echo "${rollout_exit_code}" > "${ARTIFACT_DIR}/rollout_exit_code.txt"
echo "=== rollout finished $(date -u +%Y-%m-%dT%H:%M:%SZ) ==="
exit 0

## Step 7 - Stop server and collect artifacts

Stop the upstream policy server and copy rollout logs and videos into `upstream_outputs/`.


In [ ]:
import os
import re
import shutil
import signal
import subprocess
from pathlib import Path

if 'SERVER_PID' in globals():
    try:
        os.kill(SERVER_PID, signal.SIGTERM)
    except OSError:
        pass
subprocess.run(['pkill', '-f', 'run_gr00t_server.py'], check=False)

rollout_log = (ARTIFACT_DIR / 'rollout.log').read_text(encoding='utf-8', errors='replace')
artifacts_dir = ARTIFACT_DIR / 'upstream_outputs'
artifacts_dir.mkdir(exist_ok=True)
for log_name in ('setup.log', 'policy_server.log', 'rollout.log'):
    src = ARTIFACT_DIR / log_name
    if src.is_file():
        shutil.copy2(src, artifacts_dir / log_name)
video_match = re.search(r'Video saved to:\s*(\S+)', rollout_log)
video_match = re.search(r'Video saved to:\s*(\S+)', rollout_log)
    video_src = Path(video_match.group(1))
    if video_src.is_dir():
        shutil.copytree(video_src, artifacts_dir / 'videos', dirs_exist_ok=True)

print('upstream_verdict=', (ARTIFACT_DIR / 'verdict.txt').read_text(encoding='utf-8').strip())
